# Day 2 — Trial Criteria Mapping

## Section 1 — Objective

Clinical trial eligibility criteria are written in natural language and often combine multiple clinical concepts into a single sentence.

TrialMatch AI needs to decompose those criteria before looking for patient evidence.

The eventual pipeline is:

```
Trial criterion
      ↓
Atomic criteria
      ↓
Evidence requirements
      ↓
MIMIC-IV evidence
      ↓
SATISFIED / NOT_SATISFIED / UNKNOWN
```

Today stops at **evidence requirements**. This notebook produces a human-created reference representation that we can later compare against automated NLP/LLM extraction.

**No patient evaluation happens here.** SATISFIED / NOT_SATISFIED / UNKNOWN apply to a particular patient's evidence, not to a criterion itself, so they are not assigned in this notebook.

## Section 2 — Define the reference schema

`criteria_reference` uses these columns:

| column | meaning |
|---|---|
| criterion_id | stable identifier for the atomic criterion |
| criterion_type | INCLUSION / EXCLUSION / STUDY_LEVEL |
| original_text | the supplied trial text the row was decomposed from |
| atomic_concept | one clinical concept |
| operator | comparison or presence check |
| threshold | numeric value when explicitly supplied |
| unit | unit exactly as supplied (never invented) |
| temporal_requirement | when the evidence must hold |
| qualifier | clinically meaningful wording that is not a number |
| boolean_group | which atomic rows combine (and how, when explicit) |
| potential_mimic_source | likely MIMIC-IV table(s) |
| evidence_type | diagnosis_code / procedure_code / vital_sign / laboratory_result / demographic_field / other |
| mimic_evaluability | DIRECT / PARTIAL / DIFFICULT / NOT_DIRECTLY_EVALUABLE |
| notes | ambiguity, normalization needs, preserved wording |

In [1]:
import pandas as pd

criteria_reference = pd.DataFrame(columns=[
    "criterion_id",
    "criterion_type",
    "original_text",
    "atomic_concept",
    "operator",
    "threshold",
    "unit",
    "temporal_requirement",
    "qualifier",
    "boolean_group",
    "potential_mimic_source",
    "evidence_type",
    "mimic_evaluability",
    "notes",
])

print("Schema columns:", criteria_reference.columns.tolist())
print("Evaluability values:", ["DIRECT", "PARTIAL", "DIFFICULT", "NOT_DIRECTLY_EVALUABLE"])

Schema columns: ['criterion_id', 'criterion_type', 'original_text', 'atomic_concept', 'operator', 'threshold', 'unit', 'temporal_requirement', 'qualifier', 'boolean_group', 'potential_mimic_source', 'evidence_type', 'mimic_evaluability', 'notes']
Evaluability values: ['DIRECT', 'PARTIAL', 'DIFFICULT', 'NOT_DIRECTLY_EVALUABLE']


## Section 3 — Manually decompose criteria

Decomposition rules applied here:

- One atomic row per clinical concept.
- Explicit OR relationships are preserved as boolean groups.
- Ambiguous grammar is **not** resolved by guessing — it is flagged in `notes`.
- Undefined qualifiers (`recent`, `stable`, `unstable`, `uncontrolled`, `severe`, `acute`, `symptomatic`, `case-by-case`) are preserved as qualifiers, never converted into thresholds.
- No threshold or unit is invented.

In [2]:
rows = [{'criterion_id': 'IN_01A', 'criterion_type': 'INCLUSION', 'original_text': 'History of acute coronary syndromes, had a recent revascularization procedure', 'atomic_concept': 'history of acute coronary syndrome', 'operator': 'presence (diagnosis)', 'threshold': None, 'unit': '', 'temporal_requirement': 'history (time window undefined)', 'qualifier': 'history of', 'boolean_group': 'IN_01_GROUP_UNSPECIFIED', 'potential_mimic_source': 'diagnoses_icd + d_icd_diagnoses; admissions (timing)', 'evidence_type': 'diagnosis_code', 'mimic_evaluability': 'PARTIAL', 'notes': 'Comma relationship with IN_01B is not specified (AND vs OR vs elaboration) - ambiguity preserved. ACS is a family of diagnoses.'}, {'criterion_id': 'IN_01B', 'criterion_type': 'INCLUSION', 'original_text': 'History of acute coronary syndromes, had a recent revascularization procedure', 'atomic_concept': 'recent revascularization procedure', 'operator': 'presence (procedure)', 'threshold': None, 'unit': '', 'temporal_requirement': 'recent (undefined window)', 'qualifier': 'recent', 'boolean_group': 'IN_01_GROUP_UNSPECIFIED', 'potential_mimic_source': 'procedures_icd + d_icd_procedures; admissions (timing)', 'evidence_type': 'procedure_code', 'mimic_evaluability': 'PARTIAL', 'notes': "'recent' is not defined by the trial text. Comma relationship with IN_01A ambiguous."}, {'criterion_id': 'IN_02A', 'criterion_type': 'INCLUSION', 'original_text': 'History of stable heart failure or stable angina.', 'atomic_concept': 'history of stable heart failure', 'operator': 'presence (diagnosis)', 'threshold': None, 'unit': '', 'temporal_requirement': 'history (time window undefined)', 'qualifier': 'history of; stable (undefined)', 'boolean_group': 'IN_02_OR', 'potential_mimic_source': 'diagnoses_icd + d_icd_diagnoses', 'evidence_type': 'diagnosis_code', 'mimic_evaluability': 'DIFFICULT', 'notes': "'stable' requires additional clinical/contextual evidence beyond a heart failure diagnosis code. Explicit OR with IN_02B."}, {'criterion_id': 'IN_02B', 'criterion_type': 'INCLUSION', 'original_text': 'History of stable heart failure or stable angina.', 'atomic_concept': 'history of stable angina', 'operator': 'presence (diagnosis)', 'threshold': None, 'unit': '', 'temporal_requirement': 'history (time window undefined)', 'qualifier': 'history of; stable (undefined)', 'boolean_group': 'IN_02_OR', 'potential_mimic_source': 'diagnoses_icd + d_icd_diagnoses', 'evidence_type': 'diagnosis_code', 'mimic_evaluability': 'DIFFICULT', 'notes': "'stable' requires additional clinical/contextual evidence beyond an angina diagnosis code. Explicit OR with IN_02A."}, {'criterion_id': 'EX_01A', 'criterion_type': 'EXCLUSION', 'original_text': 'Unstable angina', 'atomic_concept': 'unstable angina', 'operator': 'presence (diagnosis)', 'threshold': None, 'unit': '', 'temporal_requirement': 'at screening (active condition)', 'qualifier': 'unstable (standard clinical label; coded as its own diagnosis)', 'boolean_group': '', 'potential_mimic_source': 'diagnoses_icd + d_icd_diagnoses', 'evidence_type': 'diagnosis_code', 'mimic_evaluability': 'DIRECT', 'notes': 'Unstable angina is a standard coded diagnosis. Absence of a code is not proof of absence.'}, {'criterion_id': 'EX_02A', 'criterion_type': 'EXCLUSION', 'original_text': 'Systolic blood pressure of >200 mm Hg, or diastolic blood pressure >110 mm Hg (should be assessed on a case-by-case basis)', 'atomic_concept': 'systolic blood pressure', 'operator': '>', 'threshold': 200, 'unit': 'mm Hg', 'temporal_requirement': 'at assessment (timing and setting undefined)', 'qualifier': 'case-by-case assessment', 'boolean_group': 'EX_02_OR', 'potential_mimic_source': 'omr; chartevents + d_items (ICU vitals)', 'evidence_type': 'vital_sign', 'mimic_evaluability': 'PARTIAL', 'notes': "Explicit OR with EX_02B. 'Case-by-case' assessment preserved; measurement context not defined by trial text."}, {'criterion_id': 'EX_02B', 'criterion_type': 'EXCLUSION', 'original_text': 'Systolic blood pressure of >200 mm Hg, or diastolic blood pressure >110 mm Hg (should be assessed on a case-by-case basis)', 'atomic_concept': 'diastolic blood pressure', 'operator': '>', 'threshold': 110, 'unit': 'mm Hg', 'temporal_requirement': 'at assessment (timing and setting undefined)', 'qualifier': 'case-by-case assessment', 'boolean_group': 'EX_02_OR', 'potential_mimic_source': 'omr; chartevents + d_items (ICU vitals)', 'evidence_type': 'vital_sign', 'mimic_evaluability': 'PARTIAL', 'notes': "Explicit OR with EX_02A. 'Case-by-case' assessment preserved; measurement context not defined by trial text."}, {'criterion_id': 'EX_03A', 'criterion_type': 'EXCLUSION', 'original_text': 'Orthostatic blood pressure drop of >20 mm Hg with symptoms', 'atomic_concept': 'orthostatic blood pressure drop', 'operator': '>', 'threshold': 20, 'unit': 'mm Hg', 'temporal_requirement': "across postural change (supine-to-standing implied by 'orthostatic'; exact protocol undefined)", 'qualifier': 'with symptoms (additional evidence requirement)', 'boolean_group': '', 'potential_mimic_source': 'omr / chartevents + d_items (positional BP measurements if recorded)', 'evidence_type': 'vital_sign (paired measurements)', 'mimic_evaluability': 'DIFFICULT', 'notes': "NOT reduced to a simple BP >20 rule. Requires paired positional BP measurements AND symptom evidence; 'with symptoms' preserved as an additional evidence requirement."}, {'criterion_id': 'EX_04A', 'criterion_type': 'EXCLUSION', 'original_text': 'May have had the implantation of cardiac defibrillators (ICD)', 'atomic_concept': 'cardiac defibrillator (ICD) implantation', 'operator': 'presence (procedure/device)', 'threshold': None, 'unit': '', 'temporal_requirement': 'history (window undefined)', 'qualifier': "'may have had' - phrasing ambiguous (possibility vs confirmed history)", 'boolean_group': '', 'potential_mimic_source': 'procedures_icd + d_icd_procedures; diagnoses_icd (device status)', 'evidence_type': 'procedure_code', 'mimic_evaluability': 'PARTIAL', 'notes': "Device presence may require multi-table confirmation. 'May have had' ambiguity preserved."}, {'criterion_id': 'EX_05A', 'criterion_type': 'EXCLUSION', 'original_text': 'Synchronization devices or ventricular assist devices', 'atomic_concept': 'synchronization device', 'operator': 'presence (procedure/device)', 'threshold': None, 'unit': '', 'temporal_requirement': 'history (window undefined)', 'qualifier': 'device category is vague - requires concept normalization before implementation', 'boolean_group': 'EX_05_OR', 'potential_mimic_source': 'procedures_icd + d_icd_procedures; diagnoses_icd', 'evidence_type': 'procedure_code', 'mimic_evaluability': 'DIFFICULT', 'notes': "Explicit OR with EX_05B. 'Synchronization devices' has no explicit taxonomy in the supplied tables; no definition invented."}, {'criterion_id': 'EX_05B', 'criterion_type': 'EXCLUSION', 'original_text': 'Synchronization devices or ventricular assist devices', 'atomic_concept': 'ventricular assist device', 'operator': 'presence (procedure/device)', 'threshold': None, 'unit': '', 'temporal_requirement': 'history (window undefined)', 'qualifier': '', 'boolean_group': 'EX_05_OR', 'potential_mimic_source': 'procedures_icd + d_icd_procedures; diagnoses_icd', 'evidence_type': 'procedure_code', 'mimic_evaluability': 'PARTIAL', 'notes': 'Explicit OR with EX_05A.'}, {'criterion_id': 'EX_06A', 'criterion_type': 'EXCLUSION', 'original_text': 'Have had a heart valve repair/replacement, a heart transplant, or grown-up congenital heart disease, critical aortic stenosis, acute systemic illness or fever', 'atomic_concept': 'heart valve repair', 'operator': 'presence (procedure)', 'threshold': None, 'unit': '', 'temporal_requirement': 'history (window undefined)', 'qualifier': '', 'boolean_group': 'EX_06_OR', 'potential_mimic_source': 'procedures_icd + d_icd_procedures', 'evidence_type': 'procedure_code', 'mimic_evaluability': 'PARTIAL', 'notes': "Comma-separated list with explicit 'or' connectors - treated as OR; comma/or precedence flagged."}, {'criterion_id': 'EX_06B', 'criterion_type': 'EXCLUSION', 'original_text': 'Have had a heart valve repair/replacement, a heart transplant, or grown-up congenital heart disease, critical aortic stenosis, acute systemic illness or fever', 'atomic_concept': 'heart valve replacement', 'operator': 'presence (procedure)', 'threshold': None, 'unit': '', 'temporal_requirement': 'history (window undefined)', 'qualifier': '', 'boolean_group': 'EX_06_OR', 'potential_mimic_source': 'procedures_icd + d_icd_procedures', 'evidence_type': 'procedure_code', 'mimic_evaluability': 'PARTIAL', 'notes': "List relationship treated as OR per explicit 'or' connectors."}, {'criterion_id': 'EX_06C', 'criterion_type': 'EXCLUSION', 'original_text': 'Have had a heart valve repair/replacement, a heart transplant, or grown-up congenital heart disease, critical aortic stenosis, acute systemic illness or fever', 'atomic_concept': 'heart transplant', 'operator': 'presence (procedure)', 'threshold': None, 'unit': '', 'temporal_requirement': 'history (window undefined)', 'qualifier': '', 'boolean_group': 'EX_06_OR', 'potential_mimic_source': 'procedures_icd + d_icd_procedures; diagnoses_icd', 'evidence_type': 'procedure_code', 'mimic_evaluability': 'PARTIAL', 'notes': "List relationship treated as OR per explicit 'or' connectors."}, {'criterion_id': 'EX_06D', 'criterion_type': 'EXCLUSION', 'original_text': 'Have had a heart valve repair/replacement, a heart transplant, or grown-up congenital heart disease, critical aortic stenosis, acute systemic illness or fever', 'atomic_concept': 'grown-up congenital heart disease', 'operator': 'presence (diagnosis)', 'threshold': None, 'unit': '', 'temporal_requirement': 'history (window undefined)', 'qualifier': "'grown-up' wording needs normalization (adult congenital heart disease); kept verbatim", 'boolean_group': 'EX_06_OR', 'potential_mimic_source': 'diagnoses_icd + d_icd_diagnoses', 'evidence_type': 'diagnosis_code', 'mimic_evaluability': 'PARTIAL', 'notes': "Trial wording 'grown-up' preserved; concept normalization required before implementation."}, {'criterion_id': 'EX_06E', 'criterion_type': 'EXCLUSION', 'original_text': 'Have had a heart valve repair/replacement, a heart transplant, or grown-up congenital heart disease, critical aortic stenosis, acute systemic illness or fever', 'atomic_concept': 'critical aortic stenosis', 'operator': 'presence (diagnosis)', 'threshold': None, 'unit': '', 'temporal_requirement': 'at screening', 'qualifier': 'critical (severity undefined)', 'boolean_group': 'EX_06_OR', 'potential_mimic_source': 'diagnoses_icd + d_icd_diagnoses', 'evidence_type': 'diagnosis_code', 'mimic_evaluability': 'DIFFICULT', 'notes': "Aortic stenosis is codeable; 'critical' severity requires evidence beyond a diagnosis code."}, {'criterion_id': 'EX_06F', 'criterion_type': 'EXCLUSION', 'original_text': 'Have had a heart valve repair/replacement, a heart transplant, or grown-up congenital heart disease, critical aortic stenosis, acute systemic illness or fever', 'atomic_concept': 'acute systemic illness', 'operator': 'presence (diagnosis)', 'threshold': None, 'unit': '', 'temporal_requirement': 'acute (current episode)', 'qualifier': 'acute (undefined); extremely broad concept', 'boolean_group': 'EX_06_OR', 'potential_mimic_source': 'diagnoses_icd + d_icd_diagnoses; admissions', 'evidence_type': 'diagnosis_code', 'mimic_evaluability': 'DIFFICULT', 'notes': 'Cannot be faithfully enumerated from structured codes without an operational definition.'}, {'criterion_id': 'EX_06G', 'criterion_type': 'EXCLUSION', 'original_text': 'Have had a heart valve repair/replacement, a heart transplant, or grown-up congenital heart disease, critical aortic stenosis, acute systemic illness or fever', 'atomic_concept': 'fever', 'operator': 'presence (vital sign/diagnosis)', 'threshold': None, 'unit': '', 'temporal_requirement': 'at screening', 'qualifier': 'no threshold given in trial text - none invented', 'boolean_group': 'EX_06_OR', 'potential_mimic_source': 'chartevents + d_items (temperature); diagnoses_icd (fever codes)', 'evidence_type': 'vital_sign', 'mimic_evaluability': 'PARTIAL', 'notes': 'No numeric fever threshold is supplied by the trial text; none invented.'}, {'criterion_id': 'EX_07A', 'criterion_type': 'EXCLUSION', 'original_text': 'Uncontrolled atrial or ventricular arrhythmias, acute pericarditis or myocarditis', 'atomic_concept': 'uncontrolled atrial arrhythmia', 'operator': 'presence (diagnosis)', 'threshold': None, 'unit': '', 'temporal_requirement': 'at screening', 'qualifier': 'uncontrolled (undefined)', 'boolean_group': 'EX_07_OR', 'potential_mimic_source': 'diagnoses_icd + d_icd_diagnoses', 'evidence_type': 'diagnosis_code', 'mimic_evaluability': 'DIFFICULT', 'notes': "'uncontrolled' applies to the arrhythmias; 'acute' applies to pericarditis/myocarditis - qualifier scope preserved."}, {'criterion_id': 'EX_07B', 'criterion_type': 'EXCLUSION', 'original_text': 'Uncontrolled atrial or ventricular arrhythmias, acute pericarditis or myocarditis', 'atomic_concept': 'uncontrolled ventricular arrhythmia', 'operator': 'presence (diagnosis)', 'threshold': None, 'unit': '', 'temporal_requirement': 'at screening', 'qualifier': 'uncontrolled (undefined)', 'boolean_group': 'EX_07_OR', 'potential_mimic_source': 'diagnoses_icd + d_icd_diagnoses', 'evidence_type': 'diagnosis_code', 'mimic_evaluability': 'DIFFICULT', 'notes': "'uncontrolled' undefined; requires additional clinical evidence."}, {'criterion_id': 'EX_07C', 'criterion_type': 'EXCLUSION', 'original_text': 'Uncontrolled atrial or ventricular arrhythmias, acute pericarditis or myocarditis', 'atomic_concept': 'acute pericarditis', 'operator': 'presence (diagnosis)', 'threshold': None, 'unit': '', 'temporal_requirement': 'acute (current episode)', 'qualifier': 'acute (undefined)', 'boolean_group': 'EX_07_OR', 'potential_mimic_source': 'diagnoses_icd + d_icd_diagnoses', 'evidence_type': 'diagnosis_code', 'mimic_evaluability': 'PARTIAL', 'notes': "Diagnosis code exists; 'acute' timing requires admission context."}, {'criterion_id': 'EX_07D', 'criterion_type': 'EXCLUSION', 'original_text': 'Uncontrolled atrial or ventricular arrhythmias, acute pericarditis or myocarditis', 'atomic_concept': 'acute myocarditis', 'operator': 'presence (diagnosis)', 'threshold': None, 'unit': '', 'temporal_requirement': 'acute (current episode)', 'qualifier': 'acute (undefined)', 'boolean_group': 'EX_07_OR', 'potential_mimic_source': 'diagnoses_icd + d_icd_diagnoses', 'evidence_type': 'diagnosis_code', 'mimic_evaluability': 'PARTIAL', 'notes': "Diagnosis code exists; 'acute' timing requires admission context."}, {'criterion_id': 'EX_08A', 'criterion_type': 'EXCLUSION', 'original_text': 'Recent embolism, thrombophlebitis, resting S-T segment depression (>2 mm), uncontrolled diabetes (should be assessed in accordance with local protocol and on a case-by-case basis)', 'atomic_concept': 'recent embolism', 'operator': 'presence (diagnosis)', 'threshold': None, 'unit': '', 'temporal_requirement': 'recent (undefined window)', 'qualifier': 'recent (undefined); embolism site/type unspecified', 'boolean_group': 'EX_08_GROUP_UNSPECIFIED', 'potential_mimic_source': 'diagnoses_icd + d_icd_diagnoses; admissions (timing)', 'evidence_type': 'diagnosis_code', 'mimic_evaluability': 'DIFFICULT', 'notes': 'Comma-separated list without explicit AND/OR - relationship not specified; ambiguity preserved.'}, {'criterion_id': 'EX_08B', 'criterion_type': 'EXCLUSION', 'original_text': 'Recent embolism, thrombophlebitis, resting S-T segment depression (>2 mm), uncontrolled diabetes (should be assessed in accordance with local protocol and on a case-by-case basis)', 'atomic_concept': 'thrombophlebitis', 'operator': 'presence (diagnosis)', 'threshold': None, 'unit': '', 'temporal_requirement': 'at screening', 'qualifier': '', 'boolean_group': 'EX_08_GROUP_UNSPECIFIED', 'potential_mimic_source': 'diagnoses_icd + d_icd_diagnoses', 'evidence_type': 'diagnosis_code', 'mimic_evaluability': 'PARTIAL', 'notes': 'Comma list relationship unspecified.'}, {'criterion_id': 'EX_08C', 'criterion_type': 'EXCLUSION', 'original_text': 'Recent embolism, thrombophlebitis, resting S-T segment depression (>2 mm), uncontrolled diabetes (should be assessed in accordance with local protocol and on a case-by-case basis)', 'atomic_concept': 'resting S-T segment depression', 'operator': '>', 'threshold': 2, 'unit': 'mm', 'temporal_requirement': 'at rest (ECG context)', 'qualifier': 'resting', 'boolean_group': 'EX_08_GROUP_UNSPECIFIED', 'potential_mimic_source': 'not directly available in the supplied structured tables (ECG finding); diagnoses_icd may carry related codes', 'evidence_type': 'ecg_finding', 'mimic_evaluability': 'DIFFICULT', 'notes': 'Comma list relationship unspecified. Threshold and unit are explicit in the trial text.'}, {'criterion_id': 'EX_08D', 'criterion_type': 'EXCLUSION', 'original_text': 'Recent embolism, thrombophlebitis, resting S-T segment depression (>2 mm), uncontrolled diabetes (should be assessed in accordance with local protocol and on a case-by-case basis)', 'atomic_concept': 'uncontrolled diabetes', 'operator': 'presence (diagnosis)', 'threshold': None, 'unit': '', 'temporal_requirement': 'at screening', 'qualifier': 'uncontrolled (undefined); assessed in accordance with local protocol, case-by-case (preserved verbatim)', 'boolean_group': 'EX_08_GROUP_UNSPECIFIED', 'potential_mimic_source': 'diagnoses_icd + d_icd_diagnoses; labevents + d_labitems (glycemic markers)', 'evidence_type': 'diagnosis_code', 'mimic_evaluability': 'DIFFICULT', 'notes': "'Uncontrolled' has no explicit definition; local protocol reference preserved."}, {'criterion_id': 'EX_09A', 'criterion_type': 'EXCLUSION', 'original_text': 'Severe orthopedic conditions that would prohibit exercise', 'atomic_concept': 'severe orthopedic condition prohibiting exercise', 'operator': 'presence (diagnosis) + functional impact', 'threshold': None, 'unit': '', 'temporal_requirement': 'at screening', 'qualifier': "severe (undefined); 'would prohibit exercise' is a functional clinical judgment", 'boolean_group': '', 'potential_mimic_source': 'diagnoses_icd + d_icd_diagnoses (orthopedic codes)', 'evidence_type': 'diagnosis_code', 'mimic_evaluability': 'NOT_DIRECTLY_EVALUABLE', 'notes': "Orthopedic diagnosis codes may exist, but the functional 'would prohibit exercise' component is not representable in the supplied structured tables without manual review."}, {'criterion_id': 'EX_10A', 'criterion_type': 'EXCLUSION', 'original_text': 'Other metabolic conditions, such as acute thyroiditis, hypokalemia or hyperkalemia, hypovolemia, severe rejection (cardiac transplant recipients), triple A >4.5 cm (if under surveillance will need confirmation from vascular surgeon as long as blood pressure remains stable)', 'atomic_concept': 'acute thyroiditis', 'operator': 'presence (diagnosis)', 'threshold': None, 'unit': '', 'temporal_requirement': 'acute (current episode)', 'qualifier': 'acute (undefined)', 'boolean_group': 'EX_10_GROUP_UNSPECIFIED', 'potential_mimic_source': 'diagnoses_icd + d_icd_diagnoses; labevents + d_labitems (thyroid markers)', 'evidence_type': 'diagnosis_code', 'mimic_evaluability': 'PARTIAL', 'notes': "'Other metabolic conditions, such as ...' is an open-ended illustrative list - cannot be exhaustively enumerated; list relationships not explicit."}, {'criterion_id': 'EX_10B', 'criterion_type': 'EXCLUSION', 'original_text': 'Other metabolic conditions, such as acute thyroiditis, hypokalemia or hyperkalemia, hypovolemia, severe rejection (cardiac transplant recipients), triple A >4.5 cm (if under surveillance will need confirmation from vascular surgeon as long as blood pressure remains stable)', 'atomic_concept': 'hypokalemia', 'operator': 'presence (laboratory finding)', 'threshold': None, 'unit': '', 'temporal_requirement': 'at screening', 'qualifier': 'no potassium threshold given in trial text - none invented', 'boolean_group': 'EX_10_K_OR', 'potential_mimic_source': 'labevents + d_labitems (potassium); diagnoses_icd + d_icd_diagnoses', 'evidence_type': 'laboratory_result', 'mimic_evaluability': 'PARTIAL', 'notes': "No numeric threshold supplied; none invented. Sub-group 'hypokalemia or hyperkalemia' is explicit OR (EX_10_K_OR)."}, {'criterion_id': 'EX_10C', 'criterion_type': 'EXCLUSION', 'original_text': 'Other metabolic conditions, such as acute thyroiditis, hypokalemia or hyperkalemia, hypovolemia, severe rejection (cardiac transplant recipients), triple A >4.5 cm (if under surveillance will need confirmation from vascular surgeon as long as blood pressure remains stable)', 'atomic_concept': 'hyperkalemia', 'operator': 'presence (laboratory finding)', 'threshold': None, 'unit': '', 'temporal_requirement': 'at screening', 'qualifier': 'no potassium threshold given in trial text - none invented', 'boolean_group': 'EX_10_K_OR', 'potential_mimic_source': 'labevents + d_labitems (potassium); diagnoses_icd + d_icd_diagnoses', 'evidence_type': 'laboratory_result', 'mimic_evaluability': 'PARTIAL', 'notes': "No numeric threshold supplied; none invented. Sub-group 'hypokalemia or hyperkalemia' is explicit OR (EX_10_K_OR)."}, {'criterion_id': 'EX_10D', 'criterion_type': 'EXCLUSION', 'original_text': 'Other metabolic conditions, such as acute thyroiditis, hypokalemia or hyperkalemia, hypovolemia, severe rejection (cardiac transplant recipients), triple A >4.5 cm (if under surveillance will need confirmation from vascular surgeon as long as blood pressure remains stable)', 'atomic_concept': 'hypovolemia', 'operator': 'presence (clinical assessment)', 'threshold': None, 'unit': '', 'temporal_requirement': 'at screening', 'qualifier': '', 'boolean_group': 'EX_10_GROUP_UNSPECIFIED', 'potential_mimic_source': 'diagnoses_icd + d_icd_diagnoses; labevents + d_labitems (supporting markers)', 'evidence_type': 'clinical_status', 'mimic_evaluability': 'DIFFICULT', 'notes': 'Often a clinical assessment rather than a coded diagnosis; multi-table reasoning required.'}, {'criterion_id': 'EX_10E', 'criterion_type': 'EXCLUSION', 'original_text': 'Other metabolic conditions, such as acute thyroiditis, hypokalemia or hyperkalemia, hypovolemia, severe rejection (cardiac transplant recipients), triple A >4.5 cm (if under surveillance will need confirmation from vascular surgeon as long as blood pressure remains stable)', 'atomic_concept': 'severe rejection (cardiac transplant recipients)', 'operator': 'presence (diagnosis)', 'threshold': None, 'unit': '', 'temporal_requirement': 'at screening', 'qualifier': 'severe (undefined); context: cardiac transplant recipients', 'boolean_group': 'EX_10_GROUP_UNSPECIFIED', 'potential_mimic_source': 'diagnoses_icd + d_icd_diagnoses; procedures_icd (transplant history)', 'evidence_type': 'diagnosis_code', 'mimic_evaluability': 'DIFFICULT', 'notes': "'Severe' undefined; rejection assessment typically requires biopsy/clinical context not in structured tables."}, {'criterion_id': 'EX_10F', 'criterion_type': 'EXCLUSION', 'original_text': 'Other metabolic conditions, such as acute thyroiditis, hypokalemia or hyperkalemia, hypovolemia, severe rejection (cardiac transplant recipients), triple A >4.5 cm (if under surveillance will need confirmation from vascular surgeon as long as blood pressure remains stable)', 'atomic_concept': 'triple A maximum diameter', 'operator': '>', 'threshold': 4.5, 'unit': 'cm', 'temporal_requirement': 'at screening (surveillance context if applicable)', 'qualifier': "'triple A' requires concept normalization/interpretation; surveillance clause preserved: 'if under surveillance will need confirmation from vascular surgeon as long as blood pressure remains stable'", 'boolean_group': 'EX_10_GROUP_UNSPECIFIED', 'potential_mimic_source': 'diagnoses_icd + d_icd_diagnoses (AAA codes); diameter measurement not in supplied structured tables (imaging reports)', 'evidence_type': 'measurement', 'mimic_evaluability': 'DIFFICULT', 'notes': 'Threshold and unit explicit in trial text; the diameter measurement itself is not available in the supplied structured tables.'}, {'criterion_id': 'EX_11A', 'criterion_type': 'EXCLUSION', 'original_text': 'Tachycardia >100 beats per minute at rest, or cataracts (may commence exercise after 6 weeks)', 'atomic_concept': 'resting heart rate', 'operator': '>', 'threshold': 100, 'unit': 'beats per minute', 'temporal_requirement': 'at rest (preserved; rest protocol undefined)', 'qualifier': "at rest; 'may commence exercise after 6 weeks' preserved - the anchor event for the 6-week period is not defined by the trial text", 'boolean_group': 'EX_11_OR', 'potential_mimic_source': 'chartevents + d_items (heart rate); omr (if recorded)', 'evidence_type': 'vital_sign', 'mimic_evaluability': 'PARTIAL', 'notes': "Explicit OR with EX_11B. 'At rest' preserved; no rest protocol invented."}, {'criterion_id': 'EX_11B', 'criterion_type': 'EXCLUSION', 'original_text': 'Tachycardia >100 beats per minute at rest, or cataracts (may commence exercise after 6 weeks)', 'atomic_concept': 'cataracts', 'operator': 'presence (diagnosis)', 'threshold': None, 'unit': '', 'temporal_requirement': 'at screening', 'qualifier': '6-week exercise clause attached to the cataract branch - preserved; anchor event undefined', 'boolean_group': 'EX_11_OR', 'potential_mimic_source': 'diagnoses_icd + d_icd_diagnoses', 'evidence_type': 'diagnosis_code', 'mimic_evaluability': 'PARTIAL', 'notes': 'Explicit OR with EX_11A. Post-operative cataract status not directly represented.'}, {'criterion_id': 'SL_01', 'criterion_type': 'STUDY_LEVEL', 'original_text': 'Ages Eligible: Child, Adult, Older Adult', 'atomic_concept': 'age eligibility band (Child, Adult, Older Adult)', 'operator': 'within', 'threshold': None, 'unit': 'years (band boundaries not supplied - none invented)', 'temporal_requirement': 'at enrollment (screening)', 'qualifier': 'band boundaries not defined by trial text', 'boolean_group': '', 'potential_mimic_source': 'patients (anchor_age, anchor_year)', 'evidence_type': 'demographic_field', 'mimic_evaluability': 'DIRECT', 'notes': 'Age in MIMIC-IV is deidentified via anchor fields; band boundaries must be defined later.'}, {'criterion_id': 'SL_02', 'criterion_type': 'STUDY_LEVEL', 'original_text': 'Sexes Eligible: All', 'atomic_concept': 'sex eligibility (All)', 'operator': 'no restriction', 'threshold': None, 'unit': '', 'temporal_requirement': 'at enrollment (screening)', 'qualifier': '', 'boolean_group': '', 'potential_mimic_source': 'patients (gender)', 'evidence_type': 'demographic_field', 'mimic_evaluability': 'DIRECT', 'notes': "'All' - no filtering required; gender field available for reference."}, {'criterion_id': 'SL_03', 'criterion_type': 'STUDY_LEVEL', 'original_text': 'Accepts Healthy Volunteers: No', 'atomic_concept': 'healthy volunteer status (excluded)', 'operator': 'status', 'threshold': None, 'unit': '', 'temporal_requirement': 'at screening', 'qualifier': '', 'boolean_group': '', 'potential_mimic_source': 'not represented in the supplied MIMIC tables', 'evidence_type': 'screening_concept', 'mimic_evaluability': 'NOT_DIRECTLY_EVALUABLE', 'notes': 'Healthy-volunteer status is a screening concept outside structured EHR data; requires an operational definition.'}]

criteria_reference = pd.DataFrame(rows)

print("Atomic criteria:", len(criteria_reference))
print()
print("By criterion_type:")
print(criteria_reference["criterion_type"].value_counts().to_string())
criteria_reference

Atomic criteria: 38

By criterion_type:
criterion_type
EXCLUSION      31
INCLUSION       4
STUDY_LEVEL     3


,criterion_id,criterion_type,original_text,atomic_concept,operator,threshold,unit,temporal_requirement,qualifier,boolean_group,potential_mimic_source,evidence_type,mimic_evaluability,notes
0,IN_01A,INCLUSION,"History of acute coronary syndromes, had a rec...",history of acute coronary syndrome,presence (diagnosis),NaN,,history (time window undefined),history of,IN_01_GROUP_UNSPECIFIED,diagnoses_icd + d_icd_diagnoses; admissions (t...,diagnosis_code,PARTIAL,Comma relationship with IN_01B is not specifie...
1,IN_01B,INCLUSION,"History of acute coronary syndromes, had a rec...",recent revascularization procedure,presence (procedure),NaN,,recent (undefined window),recent,IN_01_GROUP_UNSPECIFIED,procedures_icd + d_icd_procedures; admissions ...,procedure_code,PARTIAL,'recent' is not defined by the trial text. Com...
2,IN_02A,INCLUSION,History of stable heart failure or stable angina.,history of stable heart failure,presence (diagnosis),NaN,,history (time window undefined),history of; stable (undefined),IN_02_OR,diagnoses_icd + d_icd_diagnoses,diagnosis_code,DIFFICULT,'stable' requires additional clinical/contextu...
3,IN_02B,INCLUSION,History of stable heart failure or stable angina.,history of stable angina,presence (diagnosis),NaN,,history (time window undefined),history of; stable (undefined),IN_02_OR,diagnoses_icd + d_icd_diagnoses,diagnosis_code,DIFFICULT,'stable' requires additional clinical/contextu...
4,EX_01A,EXCLUSION,Unstable angina,unstable angina,presence (diagnosis),NaN,,at screening (active condition),unstable (standard clinical label; coded as it...,,diagnoses_icd + d_icd_diagnoses,diagnosis_code,DIRECT,Unstable angina is a standard coded diagnosis....
5,EX_02A,EXCLUSION,"Systolic blood pressure of >200 mm Hg, or dias...",systolic blood pressure,>,200.0,mm Hg,at assessment (timing and setting undefined),case-by-case assessment,EX_02_OR,omr; chartevents + d_items (ICU vitals),vital_sign,PARTIAL,Explicit OR with EX_02B. 'Case-by-case' assess...
6,EX_02B,EXCLUSION,"Systolic blood pressure of >200 mm Hg, or dias...",diastolic blood pressure,>,110.0,mm Hg,at assessment (timing and setting undefined),case-by-case assessment,EX_02_OR,omr; chartevents + d_items (ICU vitals),vital_sign,PARTIAL,Explicit OR with EX_02A. 'Case-by-case' assess...
7,EX_03A,EXCLUSION,Orthostatic blood pressure drop of >20 mm Hg w...,orthostatic blood pressure drop,>,20.0,mm Hg,across postural change (supine-to-standing imp...,with symptoms (additional evidence requirement),,omr / chartevents + d_items (positional BP mea...,vital_sign (paired measurements),DIFFICULT,NOT reduced to a simple BP >20 rule. Requires ...
8,EX_04A,EXCLUSION,May have had the implantation of cardiac defib...,cardiac defibrillator (ICD) implantation,presence (procedure/device),NaN,,history (window undefined),'may have had' - phrasing ambiguous (possibili...,,procedures_icd + d_icd_procedures; diagnoses_i...,procedure_code,PARTIAL,Device presence may require multi-table confir...
9,EX_05A,EXCLUSION,Synchronization devices or ventricular assist ...,synchronization device,presence (procedure/device),NaN,,history (window undefined),device category is vague - requires concept no...,EX_05_OR,procedures_icd + d_icd_procedures; diagnoses_icd,procedure_code,DIFFICULT,Explicit OR with EX_05B. 'Synchronization devi...


## Section 4 — Map atomic concepts to MIMIC evidence

Likely MIMIC-IV evidence sources per concept family:

| concept family | MIMIC evidence |
|---|---|
| Demographics | `patients` |
| Hospitalization context/timing | `admissions` |
| Diagnoses | `diagnoses_icd` + `d_icd_diagnoses` |
| Procedures | `procedures_icd` + `d_icd_procedures` |
| Blood pressure / BMI / weight / eGFR where available | `omr` |
| Laboratory measurements | `labevents` + `d_labitems` |
| Medication orders | `prescriptions` |
| Medication administrations | `emar` + `emar_detail` |
| ICU vitals/observations | `chartevents` + `d_items` |
| ICU stay context | `icustays` |

Important: a related diagnosis existing does **not** make a criterion directly measurable. For example, a *heart failure diagnosis* is easier to find than *stable heart failure*, because "stable" requires additional clinical/contextual evidence.

In [3]:
print("Evidence type counts across atomic criteria:")
print(criteria_reference["evidence_type"].value_counts().to_string())

print()
print("Concept -> likely MIMIC source (sample of 12 rows):")
criteria_reference[["criterion_id", "atomic_concept", "potential_mimic_source"]].head(12)

Evidence type counts across atomic criteria:
evidence_type
diagnosis_code                      18
procedure_code                       7
vital_sign                           4
laboratory_result                    2
demographic_field                    2
vital_sign (paired measurements)     1
ecg_finding                          1
clinical_status                      1
measurement                          1
screening_concept                    1

Concept -> likely MIMIC source (sample of 12 rows):


,criterion_id,atomic_concept,potential_mimic_source
0,IN_01A,history of acute coronary syndrome,diagnoses_icd + d_icd_diagnoses; admissions (t...
1,IN_01B,recent revascularization procedure,procedures_icd + d_icd_procedures; admissions ...
2,IN_02A,history of stable heart failure,diagnoses_icd + d_icd_diagnoses
3,IN_02B,history of stable angina,diagnoses_icd + d_icd_diagnoses
4,EX_01A,unstable angina,diagnoses_icd + d_icd_diagnoses
5,EX_02A,systolic blood pressure,omr; chartevents + d_items (ICU vitals)
6,EX_02B,diastolic blood pressure,omr; chartevents + d_items (ICU vitals)
7,EX_03A,orthostatic blood pressure drop,omr / chartevents + d_items (positional BP mea...
8,EX_04A,cardiac defibrillator (ICD) implantation,procedures_icd + d_icd_procedures; diagnoses_i...
9,EX_05A,synchronization device,procedures_icd + d_icd_procedures; diagnoses_icd


## Section 5 — Evaluability

One of four values is assigned to each atomic criterion:

- **DIRECT** — a reasonably explicit structured measurement/record can potentially answer the atomic criterion once timing is defined.
- **PARTIAL** — MIMIC contains relevant evidence, but part of the criterion/qualifier may not be directly represented.
- **DIFFICULT** — relevant evidence may exist, but substantial temporal, contextual, multi-table, or potentially unstructured reasoning would be required.
- **NOT_DIRECTLY_EVALUABLE** — the supplied structured MIMIC tables do not provide sufficient evidence to implement the criterion faithfully without additional information/manual review.

Absence of evidence is **not** treated as evidence that a condition is absent.

In [4]:
print("Evaluability counts:")
print(criteria_reference["mimic_evaluability"].value_counts().to_string())

print()
print("Evaluability by criterion_type:")
print(pd.crosstab(criteria_reference["criterion_type"], criteria_reference["mimic_evaluability"]).to_string())

Evaluability counts:
mimic_evaluability
PARTIAL                   19
DIFFICULT                 14
DIRECT                     3
NOT_DIRECTLY_EVALUABLE     2

Evaluability by criterion_type:
mimic_evaluability  DIFFICULT  DIRECT  NOT_DIRECTLY_EVALUABLE  PARTIAL
criterion_type                                                        
EXCLUSION                  12       1                       1       17
INCLUSION                   2       0                       0        2
STUDY_LEVEL                 0       2                       1        0


## Section 6 — Quantitative criteria

In [5]:
numeric_criteria = criteria_reference[criteria_reference["threshold"].notna()]

print("Atomic criteria with explicit numeric thresholds:", len(numeric_criteria))
numeric_criteria[[
    "criterion_id",
    "atomic_concept",
    "operator",
    "threshold",
    "unit",
    "temporal_requirement",
    "potential_mimic_source",
]]

Atomic criteria with explicit numeric thresholds: 6


,criterion_id,atomic_concept,operator,threshold,unit,temporal_requirement,potential_mimic_source
5,EX_02A,systolic blood pressure,>,200.0,mm Hg,at assessment (timing and setting undefined),omr; chartevents + d_items (ICU vitals)
6,EX_02B,diastolic blood pressure,>,110.0,mm Hg,at assessment (timing and setting undefined),omr; chartevents + d_items (ICU vitals)
7,EX_03A,orthostatic blood pressure drop,>,20.0,mm Hg,across postural change (supine-to-standing imp...,omr / chartevents + d_items (positional BP mea...
24,EX_08C,resting S-T segment depression,>,2.0,mm,at rest (ECG context),not directly available in the supplied structu...
32,EX_10F,triple A maximum diameter,>,4.5,cm,at screening (surveillance context if applicable),diagnoses_icd + d_icd_diagnoses (AAA codes); d...
33,EX_11A,resting heart rate,>,100.0,beats per minute,at rest (preserved; rest protocol undefined),chartevents + d_items (heart rate); omr (if re...


## Section 7 — Qualitative/contextual criteria

Qualifiers such as `history of`, `recent`, `stable`, `unstable`, `uncontrolled`, `acute`, `severe`, `at rest`, `with symptoms`, and `case-by-case` cannot be evaluated by a numeric comparison. They require either an explicit operational definition or additional evidence from other tables — and sometimes from outside the structured data entirely. This makes them systematically harder than threshold criteria.

In [6]:
qualifier_terms = [
    "history of", "recent", "stable", "unstable", "uncontrolled",
    "acute", "severe", "at rest", "resting", "with symptoms", "case-by-case",
]

def has_qualifier(text):
    if not isinstance(text, str):
        return False
    lowered = text.lower()
    return any(term in lowered for term in qualifier_terms)

qualitative = criteria_reference[criteria_reference["qualifier"].apply(has_qualifier)]

print("Atomic criteria carrying qualitative qualifiers:", len(qualitative))
qualitative[[
    "criterion_id",
    "atomic_concept",
    "qualifier",
    "temporal_requirement",
    "mimic_evaluability",
]]

Atomic criteria carrying qualitative qualifiers: 21


,criterion_id,atomic_concept,qualifier,temporal_requirement,mimic_evaluability
0,IN_01A,history of acute coronary syndrome,history of,history (time window undefined),PARTIAL
1,IN_01B,recent revascularization procedure,recent,recent (undefined window),PARTIAL
2,IN_02A,history of stable heart failure,history of; stable (undefined),history (time window undefined),DIFFICULT
3,IN_02B,history of stable angina,history of; stable (undefined),history (time window undefined),DIFFICULT
4,EX_01A,unstable angina,unstable (standard clinical label; coded as it...,at screening (active condition),DIRECT
5,EX_02A,systolic blood pressure,case-by-case assessment,at assessment (timing and setting undefined),PARTIAL
6,EX_02B,diastolic blood pressure,case-by-case assessment,at assessment (timing and setting undefined),PARTIAL
7,EX_03A,orthostatic blood pressure drop,with symptoms (additional evidence requirement),across postural change (supine-to-standing imp...,DIFFICULT
16,EX_06F,acute systemic illness,acute (undefined); extremely broad concept,acute (current episode),DIFFICULT
18,EX_07A,uncontrolled atrial arrhythmia,uncontrolled (undefined),at screening,DIFFICULT


## Section 8 — Evidence-source summary

Which MIMIC sources would potentially be required across the criteria? Counts below are computed from `potential_mimic_source`; no multi-gigabyte tables are loaded — this notebook is about mapping, not patient retrieval.

In [7]:
sources = [
    "patients",
    "admissions",
    "diagnoses_icd",
    "d_icd_diagnoses",
    "procedures_icd",
    "d_icd_procedures",
    "omr",
    "labevents",
    "d_labitems",
    "chartevents",
    "d_items",
    "prescriptions",
    "emar",
    "emar_detail",
    "icustays",
]

def source_count(source):
    return int(criteria_reference["potential_mimic_source"].str.contains(source, regex=False).sum())

source_summary = pd.DataFrame({
    "mimic_source": sources,
    "criteria_referencing": [source_count(s) for s in sources],
})
source_summary["required_for_supplied_criteria"] = source_summary["criteria_referencing"] > 0
source_summary

,mimic_source,criteria_referencing,required_for_supplied_criteria
0,patients,2,True
1,admissions,4,True
2,diagnoses_icd,28,True
3,d_icd_diagnoses,22,True
4,procedures_icd,8,True
5,d_icd_procedures,7,True
6,omr,4,True
7,labevents,5,True
8,d_labitems,5,True
9,chartevents,5,True


## Section 9 — Demonstrate future evaluation (conceptual only)

**Example 1 — threshold group, illustrative (NOT a real MIMIC patient):**

Trial criterion:

> Systolic blood pressure >200 mm Hg **OR** Diastolic blood pressure >110 mm Hg

Possible patient evidence: `SBP = 207 mm Hg`, `DBP = 94 mm Hg`

- SBP condition → evidence supports threshold
- DBP condition → evidence does not support threshold
- OR group → exclusion condition is triggered

**Example 2 — UNKNOWN, illustrative (NOT a real MIMIC patient):**

Criterion:

> Orthostatic BP drop >20 mm Hg with symptoms

Available evidence: supine and standing BP measurements are unavailable.

Result: **UNKNOWN**

Reason: insufficient evidence to evaluate the required orthostatic change.

This is an illustrative example only, not a real MIMIC patient.

## Section 10 — Important TrialMatch principle

# ABSENCE OF EVIDENCE ≠ EVIDENCE OF ABSENCE

If TrialMatch cannot find sufficient evidence to evaluate a criterion, it should **not** automatically classify the criterion as false. Missing data may mean the test was never done, the code was never entered, or the information lives outside the structured tables.

This motivates the three-state model:

- **SATISFIED** — explicit evidence supports the criterion
- **NOT_SATISFIED** — explicit evidence contradicts the criterion
- **UNKNOWN** — insufficient evidence to decide

# Day 2 Takeaways

- Trial criteria frequently contain multiple atomic clinical concepts.
- Boolean relationships such as OR must be preserved.
- Numeric thresholds are generally easier to operationalize than subjective qualifiers.
- Terms such as "recent", "stable", "uncontrolled", and "severe" require explicit definitions or additional evidence.
- Different criteria require evidence from different MIMIC tables.
- Some criteria cannot be faithfully evaluated from structured MIMIC data.
- Missing evidence must not automatically become negative evidence.
- This manually created representation will serve as a reference for later automated criteria extraction.